In [1]:
import os
import time
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType

# Inject dependencies: Hadoop, AWS, Delta lake
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,io.delta:delta-spark_2.12:3.1.0 pyspark-shell'

# Spark Session configuration with memory configs, minio connection configs and delta configs
spark = SparkSession.builder \
    .appName("NetChaos-Medallion-Pipeline") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password@123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.maximum", "100") \
    .config("spark.driver.memory", "4g") \
    .config("spark.memory.fraction", "0.6") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "1g") \
    .config("spark.sql.shuffle.partitions", "10") \
    .getOrCreate()

# Suppress overly verbose logs in Jupyter
spark.sparkContext.setLogLevel("WARN")
print(f"Spark Session Started. UI accessible at port 4040.")

Spark Session Started. UI accessible at port 4040.


In [2]:
# This matches exactly to go producer's json output
telemetry_schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("event_id", StringType(), True),
    StructField("src_ip", StringType(), True),
    StructField("dst_ip", StringType(), True),
    StructField("bytes_sent", LongType(), True),
    StructField("latency_ms", IntegerType(), True)
])

# Storage paths in MinIO
raw_path = "s3a://telemetry-data/ingest/"
bronze_path = "s3a://telemetry-data/bronze/telemetry_delta"
silver_path = "s3a://telemetry-data/silver/telemetry_parquet"
gold_path = "s3a://telemetry-data/gold/telemetry_aggregates"
checkpoint_path = "s3a://telemetry-data/checkpoints/bronze_ingest"

In [3]:
print("Starting Incremental Raw -> Bronze Ingestion...")
start_time = time.time()

# Reading as spark stream with 500 files limit to avoid memory issues
raw_stream = spark.readStream \
    .schema(telemetry_schema) \
    .option("maxFilesPerTrigger", 500) \
    .json(raw_path)

# Writing to delta using availableNow trigger
bronze_query = raw_stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(availableNow=True) \
    .start(bronze_path)

# Block until the backlog is fully processed
bronze_query.awaitTermination()

print(f"Bronze ingestion complete! Time taken: {time.time() - start_time:.2f} seconds")

Starting Incremental Raw -> Bronze Ingestion...
Bronze ingestion complete! Time taken: 153.78 seconds


In [4]:
from pyspark.sql.functions import col, year, month, dayofmonth, to_timestamp

print("Starting Bronze -> Silver Transformation...")

# Reading the bronze delta table
df_bronze = spark.read.format("delta").load(bronze_path)

# Cleaning and Enriching - Dropping bad records, extracting partitions
df_silver = df_bronze \
    .filter(col("src_ip").isNotNull() & col("bytes_sent").isNotNull()) \
    .withColumn("parsed_timestamp", to_timestamp(col("timestamp"))) \
    .withColumn("year", year(col("parsed_timestamp"))) \
    .withColumn("month", month(col("parsed_timestamp"))) \
    .withColumn("day", dayofmonth(col("parsed_timestamp")))

# Writing to silver path as partitioned parquet
# mode("overwrite") replaces the whole silver layer. If doing this daily, we should use mode("append") with dynamic partition overwrite.
df_silver.write \
    .mode("overwrite") \
    .partitionBy("year", "month", "day") \
    .parquet(silver_path)

print("Silver Parquet generation complete. Data is now optimized for analytical reads.")

Starting Bronze -> Silver Transformation...
Silver Parquet generation complete. Data is now optimized for analytical reads.


In [5]:
from pyspark.sql.functions import sum, count

print("Starting Silver -> Gold Aggregation...")

# Reading the optimized silver parquet data
df_optimized = spark.read.parquet(silver_path)

# Business Logic: Daily IP Traffic Aggregation
df_gold = df_optimized \
    .groupBy("year", "month", "day", "src_ip") \
    .agg(
        sum("bytes_sent").alias("total_bytes_transferred"),
        count("event_id").alias("total_connections")
    )

# Writing to Gold as Delta (Enables ACID transactions for BI tools)
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path)

print("Gold aggregation complete!")

# Validation
print("\n--- Top 5 Talkative IPs in Gold Layer ---")
spark.read.format("delta").load(gold_path) \
    .orderBy(col("total_bytes_transferred").desc()) \
    .show(5, truncate=False)

Starting Silver -> Gold Aggregation...
Gold aggregation complete!

--- Top 5 Talkative IPs in Gold Layer ---
+----+-----+---+-------------+-----------------------+-----------------+
|year|month|day|src_ip       |total_bytes_transferred|total_connections|
+----+-----+---+-------------+-----------------------+-----------------+
|2026|2    |22 |172.16.0.50  |309155078521           |929869           |
|2026|2    |22 |192.168.0.10 |306267164956           |929551           |
|2026|2    |22 |10.0.3.44    |306113887460           |926525           |
|2026|2    |22 |192.168.1.100|306094454071           |930504           |
|2026|2    |22 |172.16.2.20  |305170722072           |927566           |
+----+-----+---+-------------+-----------------------+-----------------+
only showing top 5 rows

